In [1]:
pip install statsbombpy azure-eventhub

  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [idna]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [azure-eventhub]m [azure-eventhub]s]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import time
from statsbombpy import sb
from azure.eventhub import EventHubProducerClient, EventData

# --- Configuration ---
# Paste your Eventstream Custom Endpoint credentials here
CONNECTION_STR = "Endpoint=sb://esehpn7zlqe2awookxa05j.servicebus.windows.net/;SharedAccessKeyName=key_beded9be-279d-41e9-a1bb-d8a87acb1570;SharedAccessKey=ORC+N1j6e4Vw/khrmSW9RpTqVxExaIbTw+AEhMzsJK4=;EntityPath=esehpn7zlqe2awookxa05j_eh"
EVENT_HUB_NAME = "esehpn7zlqe2awookxa05j_eh"

# 1. Fetch FA Cup Competition and Available Matches
print("Fetching FA Cup data from StatsBomb Open Data...")
competitions = sb.competitions()
fa_cup = competitions[competitions['competition_name'] == "FA Cup"]

if fa_cup.empty:
    print("FA Cup not found in registry; defaulting to general event extraction.")
    match_id = 3795220 # Arsenal vs Chelsea FA Cup Final
else:
    comp_id = int(fa_cup.iloc[0]['competition_id'])
    season_id = int(fa_cup.iloc[0]['season_id'])
    matches = sb.matches(competition_id=comp_id, season_id=season_id)
    match_id = int(matches.iloc[0]['match_id'])
    home_team = matches.iloc[0]['home_team']
    away_team = matches.iloc[0]['away_team']
    print(f"Streaming FA Cup Fixture: {home_team} vs {away_team} (ID: {match_id})")

# 2. Extract Event Telemetry
events_df = sb.events(match_id=match_id)
cols = ['minute', 'second', 'team', 'player', 'type', 'location', 'shot_statsbomb_xg', 'shot_outcome', 'pass_recipient']
events_clean = events_df[[c for c in cols if c in events_df.columns]].dropna(subset=['location'])

producer = EventHubProducerClient.from_connection_string(CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)
print(f"Loaded {len(events_clean)} real FA Cup match events. Streaming to Fabric...")

batch = producer.create_batch()
event_counter = 0

try:
    for _, row in events_clean.iterrows():
        loc = row['location'] if isinstance(row['location'], list) else [0.0, 0.0]
        
        payload = {
            "match_id": f"FACUP_{match_id}",
            "minute": int(row['minute']),
            "second": int(row['second']),
            "team": str(row['team']),
            "player": str(row['player']) if row['player'] == row['player'] else "Unknown",
            "event_type": str(row['type']),
            "x_coord": float(loc[0]),
            "y_coord": float(loc[1]),
            "xg": float(row['shot_statsbomb_xg']) if 'shot_statsbomb_xg' in row and row['shot_statsbomb_xg'] == row['shot_statsbomb_xg'] else 0.0,
            "shot_outcome": str(row['shot_outcome']) if 'shot_outcome' in row and row['shot_outcome'] == row['shot_outcome'] else None,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        
        batch.add(EventData(json.dumps(payload)))
        event_counter += 1
        
        # Flush every 15 events to simulate live match pacing
        if len(batch) >= 15:
            producer.send_batch(batch)
            print(f"Sent match event block at {row['minute']}:{row['second']} | Last Player: {payload['player']}")
            batch = producer.create_batch()
            time.sleep(1.5)

    if len(batch) > 0:
        producer.send_batch(batch)
        print("Final event batch sent successfully.")

except KeyboardInterrupt:
    print("Stream stopped by user.")
finally:
    producer.close()

/home/trusted-service-user/jupyter-env/python3.12/lib/python3.12/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


FA Cup not found in registry; defaulting to general event extraction.
Loaded 4782 real FA Cup match events. Streaming to Fabric...
Sent match event block at 1:7 | Last Player: Jordi Alba Ramos
Sent match event block at 2:1 | Last Player: Leonardo Bonucci
Sent match event block at 3:46 | Last Player: Aymeric Laporte
Sent match event block at 4:48 | Last Player: Nicolò Barella
Sent match event block at 5:48 | Last Player: Ferrán Torres García
Sent match event block at 6:39 | Last Player: Jordi Alba Ramos
Sent match event block at 7:46 | Last Player: Sergio Busquets i Burgos
Sent match event block at 8:42 | Last Player: Jorge Resurrección Merodio
Sent match event block at 9:39 | Last Player: Jorge Resurrección Merodio
Sent match event block at 10:26 | Last Player: Mikel Oyarzabal Ugarte
Sent match event block at 11:14 | Last Player: Sergio Busquets i Burgos
Sent match event block at 12:9 | Last Player: Emerson Palmieri dos Santos
Sent match event block at 12:40 | Last Player: Sergio Busqu